# Pairs fOU 02 — the thesis pipeline on construction-tied spreads (Binance)

Protocol: `discussions/Pairs_fOU/003_claude_pf02_basis_preregistration.md`.

**SP**: perpetual − spot (≈30 assets). **PQ**: front quarterly − perpetual (where quarterlies exist). β = 1 by construction; ADF (1%) replaces Engle–Granger; otherwise the PF01 fOU engine unchanged. Taker costs: spot 11 bp, futures 6 bp per side per leg; real funding on the perp leg.

Long runtime (1–3 h). Per-asset results are checkpointed in `02/parts/`, so a re-run resumes.

In [1]:
import os,sys,json,time,importlib,subprocess
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'));load_dotenv(ROOT/'.env')
from data_engineering.modeling import fou_pairs as fp, basis_fou as bf
for m in (fp,bf):importlib.reload(m)
assert bf.VERSION=='PF02-0.1.0' and fp.VERSION.startswith('PF01-0.1.1')
spec=fp.Spec();prm=bf.Params()
SYNTHETIC=os.getenv('PF02_SYNTHETIC','0')=='1'
if SYNTHETIC:
    from tempfile import TemporaryDirectory
    tmp=TemporaryDirectory(prefix='pf02-');OUT=Path(tmp.name)/'02';spec=fp.Spec(paths=800,structural_paths=400,n_boot=500)
else:
    OUT=ROOT/'notebooks'/'Pairs_fOU'/'02'
(OUT/'parts').mkdir(parents=True,exist_ok=True)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_basis_fou.py'),str(ROOT/'tests'/'test_fou_pairs.py')],cwd=str(ROOT),env=env,capture_output=True,text=True)
print(chk.stdout[-800:]);print(chk.stderr[-500:]);assert chk.returncode==0,'tests failed'
print(json.dumps(bf.registration(),indent=1)[:1500])

..............                                                           [100%]
14 passed in 20.85s


{
 "version": "PF02-0.1.0",
 "universe": [
  "BTC",
  "ETH",
  "SOL",
  "XRP",
  "BNB",
  "DOGE",
  "ADA",
  "LINK",
  "DOT",
  "LTC",
  "BCH",
  "UNI",
  "NEAR",
  "APT",
  "ARB",
  "OP",
  "SUI",
  "FIL",
  "ATOM",
  "ETC",
  "INJ",
  "AAVE",
  "TON",
  "WIF",
  "HBAR",
  "XLM",
  "SEI",
  "TIA",
  "ENA",
  "WLD"
 ],
 "adf_p": 0.01,
 "spot_cost_bp": 11.0,
 "fut_cost_bp": 6.0,
 "roll_days_before_expiry": 14,
 "status": "HISTORICAL_EXPLORATORY",
 "engine": {
  "version": "PF01-0.1.1",
  "assets": [
   "BTC",
   "ETH",
   "SOL",
   "XRP"
  ],
  "formation_bars": 2016,
  "entry_z": 1.5,
  "target_p": 0.7,
  "max_horizon": 48,
  "structural_horizon": 252,
  "memory": 60,
  "paths": 5000,
  "structural_paths": 2000,
  "coint_p": 0.01,
  "adf_p": 0.05,
  "cost_bp_per_side_leg": 6.0,
  "test_start": "2025-10-01",
  "n_boot": 4000,
  "seed": 42,
  "status": "HISTORICAL_EXPLORATORY"
 }
}


In [2]:
def load_inputs(A):
    """Returns {'SP': (panel, funding_y, funding_x, cy, cx), 'PQ': ...} for one asset."""
    if SYNTHETIC:
        P,f=bf.synthetic_basis(n_days=12,seed=hash(A)%100,kappa=0.08,sd_bp=25)
        P.columns=[f'PERP_{A}',f'SPOT_{A}']
        return {'SP':(P,f,None,prm.fut_cost_bp,prm.spot_cost_bp)}
    sym=f'{A}USDT';out={}
    pp=BIN/f'{sym}_um_1m.parquet';sp=BIN/'basis'/'spot'/f'{sym}_spot_1m.parquet';fu=BIN/'basis'/'funding'/f'{sym}_funding.parquet'
    if not pp.exists():return out
    perp=fp.log_close_5m(pd.read_parquet(pp,columns=['open_time','close']))
    fund=bf.funding_series(pd.read_parquet(fu)) if fu.exists() else None
    if sp.exists():
        spot=fp.log_close_5m(pd.read_parquet(sp,columns=['open_time','close']))
        out['SP']=(fp.panel({f'PERP_{A}':perp,f'SPOT_{A}':spot}),fund,None,prm.fut_cost_bp,prm.spot_cost_bp)
    qfiles=sorted((BIN/'basis'/'quarterly').glob(f'{sym}_*_1m.parquet'))
    if qfiles:
        con={}
        for q in qfiles:
            e=pd.Timestamp('20'+q.name.split('_')[1],tz='UTC')+pd.Timedelta(hours=8)
            con[e]=fp.log_close_5m(pd.read_parquet(q,columns=['open_time','close']))
        front=bf.front_series(con,prm.roll_days_before_expiry)
        out['PQ']=(fp.panel({f'QTR_{A}':front,f'PERP_{A}':perp}).dropna(how='all'),None,fund,prm.fut_cost_bp,prm.fut_cost_bp)
    return out
if not SYNTHETIC:
    BIN=Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent/'Binance'
ASSETS=['AAA','BBB'] if SYNTHETIC else list(prm.universe)
print('assets:',ASSETS)

assets: ['BTC', 'ETH', 'SOL', 'XRP', 'BNB', 'DOGE', 'ADA', 'LINK', 'DOT', 'LTC', 'BCH', 'UNI', 'NEAR', 'APT', 'ARB', 'OP', 'SUI', 'FIL', 'ATOM', 'ETC', 'INJ', 'AAVE', 'TON', 'WIF', 'HBAR', 'XLM', 'SEI', 'TIA', 'ENA', 'WLD']


## Run (checkpointed per asset and family)

In [3]:
T0=time.time();scale=[]
for A in ASSETS:
    inp=load_inputs(A)
    if not inp:print(A,'no perp file - skipped');continue
    for fam,(P,fy_,fx_,cy,cx) in inp.items():
        y,x=P.columns[0],P.columns[1]
        part=OUT/'parts'/f'{fam}_{A}'
        s=bf.deviation_scale(P,y,x);scale.append({'family':fam,'asset':A,'median_7d_sd_bp':float(s.median()),
             'p90_7d_sd_bp':float(s.quantile(.9)),'bars':int(P.notna().all(axis=1).sum())})
        if (part.with_suffix('.fits.parquet')).exists():
            print(f'{fam} {A}: cached');continue
        cache={};TR=[];FC=[];FI=[]
        for mode in ('fou','ou','zscore'):
            tr,fc,fi=fp.run_pair(P,y,x,spec,mode=mode,screen_cache=cache,screen_fn=bf.screen_basis,
                                 funding_y=fy_,funding_x=fx_,cost_y_bp=cy,cost_x_bp=cx,label=f'{fam}_{A}')
            TR.append(tr);FC.append(fc);FI.append(fi)
        tr=pd.concat([t for t in TR if len(t)],ignore_index=True) if any(len(t) for t in TR) else pd.DataFrame()
        fc=pd.concat([f for f in FC if len(f)],ignore_index=True) if any(len(f) for f in FC) else pd.DataFrame()
        fi=pd.concat(FI,ignore_index=True)
        if len(tr):tr.to_parquet(part.with_suffix('.trades.parquet'),index=False)
        if len(fc):fc.drop(columns=[c for c in fc.columns if fc[c].dtype==object and c not in('pair','mode')],errors='ignore').to_parquet(part.with_suffix('.fc.parquet'),index=False)
        fi.drop(columns=['resid'],errors='ignore').to_parquet(part.with_suffix('.fits.parquet'),index=False)
        print(f'{fam} {A}: trades={len(tr)} eligible fou days={int(fi[fi["mode"]=="fou"].eligible.sum())} | {time.time()-T0:6.0f}s',flush=True)
SCALE=pd.DataFrame(scale);display(SCALE.round(2))

SP BTC: trades=33417 eligible fou days=619 |    175s
PQ BTC: trades=4008 eligible fou days=63 |    222s
SP ETH: trades=31293 eligible fou days=639 |    390s
PQ ETH: trades=7154 eligible fou days=100 |    452s
SP SOL: trades=27017 eligible fou days=517 |    605s
SP XRP: trades=22846 eligible fou days=492 |    744s
SP BNB: trades=9692 eligible fou days=257 |    850s
SP DOGE: trades=34655 eligible fou days=650 |   1032s
SP ADA: trades=38862 eligible fou days=604 |   1220s
SP LINK: trades=53181 eligible fou days=718 |   1441s
SP DOT: trades=45260 eligible fou days=684 |   1640s
SP LTC: trades=41791 eligible fou days=691 |   1829s
SP BCH: trades=33396 eligible fou days=542 |   2002s
SP UNI: trades=45385 eligible fou days=701 |   2349s
SP NEAR: trades=43055 eligible fou days=734 |   2603s
SP APT: trades=45545 eligible fou days=649 |   2872s
SP ARB: trades=48620 eligible fou days=727 |   3149s
SP OP: trades=49121 eligible fou days=738 |   3380s
SP SUI: trades=33142 eligible fou days=609 |   3

,family,asset,median_7d_sd_bp,p90_7d_sd_bp,bars
0,SP,BTC,0.84,1.83,213408
1,PQ,BTC,6.63,18.71,213400
2,SP,ETH,0.93,1.59,213408
3,PQ,ETH,6.56,18.67,213400
4,SP,SOL,1.50,2.85,213408
5,SP,XRP,1.38,3.09,213408
6,SP,BNB,2.59,4.42,231840
7,SP,DOGE,1.43,2.26,231840
8,SP,ADA,2.72,4.89,231840
9,SP,LINK,4.09,5.87,231840


In [4]:
def rd(sfx):
    fs=sorted((OUT/'parts').glob(f'*.{sfx}.parquet'))
    return pd.concat([pd.read_parquet(f) for f in fs],ignore_index=True) if fs else pd.DataFrame()
trades,fcs,fits=rd('trades'),rd('fc'),rd('fits')
for d in (trades,fits,fcs):
    if len(d):d['family']=d['pair'].str.split('_').str[0]
TS=pd.Timestamp(spec.test_start,tz='UTC')
print('Formation funnel (fOU):');display(fits[fits['mode']=='fou'].groupby(['family','reason']).size().unstack(fill_value=0))
if len(trades):trades['period']=np.where(trades.entry_bar>=TS,'test','dev')
rows=[]
for fam in sorted(trades.family.unique()) if len(trades) else []:
    for mode in ('fou','ou','zscore'):
        for per in ('dev','test'):
            t=trades[(trades.family==fam)&(trades['mode']==mode)&(trades.period==per)]
            rows.append(fp.trade_summary(t,spec,f'{fam}|{mode}|{per}|all'))
            if mode=='fou':
                for side,g in t.groupby('side'):rows.append(fp.trade_summary(g,spec,f'{fam}|{mode}|{per}|side={side}'))
                for a,g in t.groupby('pair'):rows.append(fp.trade_summary(g,spec,f'{fam}|{mode}|{per}|{a}'))
summ=pd.DataFrame(rows)
display(summ[~summ.label.str.contains(r'\|(?:SP|PQ)_')].round(2))
print('Per asset (fOU, TEST):');display(summ[summ.label.str.contains('fou\\|test\\|(?:SP|PQ)_')].round(2))
if len(trades):print('Mean funding and cost per trade (bp of gross notional):');display(trades.groupby(['family','mode'])[['gross_bp','funding_bp','cost_bp','net_bp']].mean().round(2))

Formation funnel (fOU):


reason,,H>=0.5,gaps,no_coint,params,structural
family,,,,,,
PQ,163,0,490,1160,0,33
SP,19377,46,395,2868,34,1245


,label,trades,days_with_trades,gross_bp,gross_lo,gross_hi,net_bp,net_lo,net_hi,win_rate_net,converge_share,mean_bars_held
0,PQ|fou|dev|all,89,8,-0.94,-3.61,0.29,-12.94,-15.62,-11.71,0.0,0.53,7.28
1,PQ|fou|dev|side=long_y,49,7,-0.31,-0.84,0.08,-12.29,-12.83,-11.90,0.0,0.53,7.31
2,PQ|fou|dev|side=short_y,40,6,-1.70,-6.97,1.01,-13.74,-19.01,-11.05,0.0,0.52,7.25
5,PQ|fou|test|all,2633,123,0.02,-0.09,0.14,-11.98,-12.10,-11.86,0.0,0.57,5.19
6,PQ|fou|test|side=long_y,1751,115,0.06,-0.06,0.19,-11.94,-12.06,-11.80,0.0,0.56,5.55
7,PQ|fou|test|side=short_y,882,101,-0.07,-0.24,0.10,-12.08,-12.25,-11.91,0.0,0.60,4.48
10,PQ|ou|dev|all,572,24,-0.23,-0.65,0.08,-12.23,-12.65,-11.93,0.0,0.08,5.47
11,PQ|ou|test|all,5371,140,0.02,-0.04,0.08,-11.98,-12.04,-11.92,0.0,0.19,1.52
12,PQ|zscore|dev|all,182,22,-0.57,-1.90,0.18,-12.55,-13.90,-11.77,0.0,0.52,28.40
13,PQ|zscore|test|all,2315,135,0.08,-0.03,0.19,-11.92,-12.04,-11.81,0.0,0.93,10.30


Per asset (fOU, TEST):


,label,trades,days_with_trades,gross_bp,gross_lo,gross_hi,net_bp,net_lo,net_hi,win_rate_net,converge_share,mean_bars_held
8,PQ|fou|test|PQ_BTC,913,63,0.03,-0.11,0.16,-11.97,-12.11,-11.84,0.00,0.57,5.29
9,PQ|fou|test|PQ_ETH,1720,93,0.01,-0.14,0.18,-11.99,-12.14,-11.83,0.00,0.57,5.13
50,SP|fou|test|SP_AAVE,5943,345,0.22,0.19,0.25,-16.78,-16.81,-16.75,0.00,0.61,4.58
51,SP|fou|test|SP_ADA,7939,331,0.01,-0.04,0.06,-16.98,-17.04,-16.93,0.00,0.59,3.09
52,SP|fou|test|SP_APT,7360,307,0.19,0.06,0.36,-16.80,-16.93,-16.63,0.00,0.60,2.39
53,SP|fou|test|SP_ARB,9198,371,-0.05,-0.13,0.03,-17.05,-17.13,-16.97,0.00,0.54,2.12
54,SP|fou|test|SP_ATOM,8218,356,0.25,0.18,0.32,-16.74,-16.81,-16.67,0.00,0.62,2.50
55,SP|fou|test|SP_BCH,5151,254,0.10,0.05,0.14,-16.89,-16.94,-16.85,0.00,0.60,3.25
56,SP|fou|test|SP_BNB,605,84,0.22,0.17,0.28,-16.76,-16.82,-16.70,0.00,0.58,10.69
57,SP|fou|test|SP_BTC,6143,351,0.10,0.08,0.11,-16.90,-16.92,-16.88,0.00,0.62,4.59


Mean funding and cost per trade (bp of gross notional):


gross_bp  funding_bp  cost_bp  net_bp
family mode                                         
PQ     fou        -0.01       -0.00     12.0  -12.01
       ou         -0.01        0.00     12.0  -12.01
       zscore      0.03        0.00     12.0  -11.97
SP     fou         0.18        0.01     17.0  -16.81
       ou          0.12        0.00     17.0  -16.88
       zscore      0.33        0.02     17.0  -16.65

In [5]:
cal=[]
for fam in sorted(fcs.family.unique()) if len(fcs) else []:
    for mode in ('fou','ou'):
        for per in ('dev','test'):
            f=fcs[(fcs.family==fam)&(fcs['mode']==mode)];f=f[(f.signal_bar>=TS)==(per=='test')]
            cal.append({'family':fam,'mode':mode,'period':per,**fp.calibration(f)})
cal=pd.DataFrame(cal);display(cal.round(4))
def decide(summ):
    if SYNTHETIC:return {'verdict':'SYNTHETIC_NO_MARKET_INFERENCE'}
    out={}
    for fam in ('SP','PQ'):
        r=summ[summ.label==f'{fam}|fou|test|all']
        if r.empty or not r.iloc[0].get('trades',0):out[fam]={'verdict':'NO_TRADES'};continue
        r=r.iloc[0]
        v=('NET_EDGE_CANDIDATE' if r.net_lo>0 else 'GROSS_EDGE_ONLY' if r.gross_lo>0 else
           'NO_EDGE' if (r.net_hi<0) else 'INCONCLUSIVE')
        out[fam]={'verdict':v,'trades':int(r.trades),'gross':[r.gross_bp,r.gross_lo,r.gross_hi],'net':[r.net_bp,r.net_lo,r.net_hi]}
    out['primary']='SP';out['status']=prm.status;return out
dec=decide(summ);print(json.dumps(dec,indent=2,default=float))
summ.to_csv(OUT/'02_summary.csv',index=False);cal.to_csv(OUT/'02_calibration.csv',index=False);SCALE.to_csv(OUT/'02_basis_scale.csv',index=False)
fits.to_csv(OUT/'02_formation.csv',index=False,float_format='%.6g')
if len(trades):trades.to_csv(OUT/'02_trades.csv.gz',index=False,float_format='%.6g')
(OUT/'02_decision.json').write_text(json.dumps(dec,indent=2,default=float),encoding='utf-8')
(OUT/'02_manifest.json').write_text(json.dumps({'registration':bf.registration(),'synthetic':SYNTHETIC,'runtime_s':time.time()-T0},indent=2,default=str),encoding='utf-8')
print('saved to',OUT)

,family,mode,period,forecasts
0,PQ,fou,dev,0
1,PQ,fou,test,0
2,PQ,ou,dev,0
3,PQ,ou,test,0
4,SP,fou,dev,0
5,SP,fou,test,0
6,SP,ou,dev,0
7,SP,ou,test,0


{
  "SP": {
    "verdict": "GROSS_EDGE_ONLY",
    "trades": 202343,
    "gross": [
      0.15946221534327346,
      0.14054744091920843,
      0.18121894514608877
    ],
    "net": [
      -16.833267835115603,
      -16.852325890088242,
      -16.811364609010234
    ]
  },
  "PQ": {
    "verdict": "NO_EDGE",
    "trades": 2633,
    "gross": [
      0.017981744235519403,
      -0.09473914361249129,
      0.1365358447996251
    ],
    "net": [
      -11.983339733166682,
      -12.095547669236229,
      -11.86423670452539
    ]
  },
  "primary": "SP",
  "status": "HISTORICAL_EXPLORATORY"
}
saved to D:\Projects\deribit_market_data\notebooks\Pairs_fOU\02
